# Baseline Music Generation
This notebook implements two baselines:
1. **Random Note Generator**: Uniformly samples pitches.
2. **Markov Chain**: First-order transition matrix from training data.

In [ ]:
import numpy as np
import os
from pathlib import Path
import sys
import pretty_midi

# Add src to path
sys.path.append(str(Path(os.getcwd()).parent))

from src.config import GENERATED_MIDI_DIR, PROCESSED_DATA_DIR, FS, SEQ_LEN
from src.preprocessing.piano_roll import piano_roll_to_pretty_midi

## Baseline 1: Random Note Generator

In [ ]:
def generate_random_music(num_steps=64, num_samples=5):
    for i in range(num_samples):
        # Random binary matrix (128, num_steps)
        # Sparsity: only a few notes at a time
        roll = (np.random.rand(128, num_steps) > 0.98).astype(np.float32)
        pm = piano_roll_to_pretty_midi(roll, fs=FS)
        output_path = GENERATED_MIDI_DIR / f"baseline_random_{i}.mid"
        pm.write(str(output_path))
        print(f"Saved {output_path}")

generate_random_music()

## Baseline 2: Markov Chain

In [ ]:
def train_markov_chain():
    # Load training data
    train_data = np.load(PROCESSED_DATA_DIR / "maestro_train.npy") # (N, 64, 128)
    
    # First-order Markov chain on individual pitches or on the whole 128-dim vector?
    # Let's do a simple one: transition between pitch sets (multi-hot vectors converted to tuple/int)
    # But that might be too sparse. Let's do pitch-wise Markov or just sample from pitch distribution.
    
    # Better: Markov chain on the tokens if we had them. 
    # For piano-roll, let's just count transitions between "active pitch sets".
    
    transitions = {}
    
    print("Training Markov Chain...")
    # Take a subset if too large
    subset = train_data[:1000]
    
    for seq in subset:
        # seq is (64, 128)
        for t in range(len(seq) - 1):
            curr = tuple(seq[t])
            nxt = tuple(seq[t+1])
            if curr not in transitions:
                transitions[curr] = {}
            transitions[curr][nxt] = transitions[curr].get(nxt, 0) + 1
            
    # Normalize
    for curr in transitions:
        total = sum(transitions[curr].values())
        for nxt in transitions[curr]:
            transitions[curr][nxt] /= total
            
    return transitions

def generate_markov_music(transitions, num_steps=64, num_samples=5):
    all_keys = list(transitions.keys())
    
    for i in range(num_samples):
        curr = all_keys[np.random.randint(len(all_keys))]
        roll_seq = [np.array(curr)]
        
        for _ in range(num_steps - 1):
            if curr in transitions:
                options = list(transitions[curr].keys())
                probs = list(transitions[curr].values())
                idx = np.random.choice(len(options), p=probs)
                curr = options[idx]
            else:
                curr = all_keys[np.random.randint(len(all_keys))]
            roll_seq.append(np.array(curr))
            
        roll = np.stack(roll_seq).T # (128, 64)
        pm = piano_roll_to_pretty_midi(roll, fs=FS)
        output_path = GENERATED_MIDI_DIR / f"baseline_markov_{i}.mid"
        pm.write(str(output_path))
        print(f"Saved {output_path}")

transitions = train_markov_chain()
generate_markov_music(transitions)